# Bước 3: Trích xuất đặc trưng ảnh cá trên Kaggle

**Đầu vào**: Dataset chứa `images.zip` hoặc thư mục `images/` có `0000.png` đến `0999.png`.

**Đầu ra**: `fish.db`, `feature_matrices.npz`, CSV kiểm thử, hình minh họa, báo cáo JSON.

Notebook này **tự chứa mã nguồn project** để chạy được khi chỉ gắn Dataset ảnh. Không cần GPU. Các vector 671 chiều là cấu hình thử nghiệm; chưa đánh giá tìm kiếm theo thị giác người.

Sau khi chạy, vào tab **Output** của Notebook để lưu/tải các file hoặc sử dụng **Save Version**.

In [ ]:
from pathlib import Path
import sys, subprocess, zipfile, json, os
import numpy as np
WORK=Path('/kaggle/working/fish_step3') if Path('/kaggle/working').exists() else Path('./fish_step3_kaggle')
WORK.mkdir(parents=True,exist_ok=True)
print('Project folder:',WORK)
# Kaggle usually has these libraries; uncomment next line only if a package is missing:
# subprocess.run([sys.executable,'-m','pip','install','opencv-python-headless','scikit-image','scipy','matplotlib'],check=True)

In [ ]:
p=WORK/'fishcbir/__init__.py'
p.parent.mkdir(parents=True,exist_ok=True)
p.write_text('',encoding='utf-8')
print('Created',p)

In [ ]:
p=WORK/'fishcbir/preprocessing.py'
p.parent.mkdir(parents=True,exist_ok=True)
p.write_text('"""White-background fish segmentation and contour extraction.\n\nForeground is extracted from non-nearly-white pixels; largest component is filled\nso bright fish body regions are not mistaken for background. All features consume\nexactly the same foreground mask. Errors are reported, not silently hidden.\n"""\nfrom dataclasses import dataclass\nimport cv2\nimport numpy as np\n\n@dataclass\nclass PreparedImage:\n    rgb: np.ndarray\n    gray: np.ndarray\n    hsv: np.ndarray\n    mask: np.ndarray\n    contour: np.ndarray\n    bbox: tuple[int, int, int, int]\n    area: int\n    mask_ratio: float\n    status: str\n\n\ndef preprocess(rgb: np.ndarray, whiteness_threshold: int = 250) -> PreparedImage:\n    if rgb is None or rgb.ndim != 3 or rgb.shape[2] != 3:\n        raise ValueError(\'Expected an H×W×3 RGB image\')\n    if rgb.dtype != np.uint8:\n        raise ValueError(\'Expected uint8 RGB image\')\n\n    h, w = rgb.shape[:2]\n    raw = np.any(rgb < whiteness_threshold, axis=2).astype(np.uint8) * 255\n    raw = cv2.morphologyEx(raw, cv2.MORPH_CLOSE,\n                           cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)))\n    contours, _ = cv2.findContours(raw, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)\n    if not contours:\n        raise ValueError(\'Fish silhouette not found; all pixels are near-white\')\n    contour = max(contours, key=cv2.contourArea)\n    mask = np.zeros((h, w), dtype=np.uint8)\n    cv2.drawContours(mask, [contour], -1, 255, thickness=cv2.FILLED)\n    # Thin detached tips may not belong to main contour; do not artificially attach them.\n    ys, xs = np.nonzero(mask)\n    if len(xs) < 20:\n        raise ValueError(\'Fish mask too small\')\n    x0, x1 = int(xs.min()), int(xs.max()+1)\n    y0, y1 = int(ys.min()), int(ys.max()+1)\n    area = int(len(xs))\n    ratio = area/(h*w)\n    flags=[]\n    if ratio < .04 or ratio > .8: flags.append(\'unusual_area\')\n    if x0 == 0 or y0 == 0 or x1 == w or y1 == h: flags.append(\'touches_border\')\n    # Warn when raw foreground is poorly retained by the main contour.\n    raw_n=int((raw>0).sum())\n    if raw_n and int(np.logical_and(raw>0,mask>0).sum())/raw_n < .9:\n        flags.append(\'disconnected_foreground\')\n    gray=cv2.cvtColor(rgb,cv2.COLOR_RGB2GRAY)\n    hsv=cv2.cvtColor(rgb,cv2.COLOR_RGB2HSV)\n    return PreparedImage(rgb,gray,hsv,mask>0,contour,(x0,y0,x1,y1),area,ratio,\n                         \'ok\' if not flags else \',\'.join(flags))\n',encoding='utf-8')
print('Created',p)

In [ ]:
p=WORK/'fishcbir/features.py'
p.parent.mkdir(parents=True,exist_ok=True)
p.write_text('"""Eight interpretable fish image descriptors, aligned with report Part 2.\n\nDimensions (default): GLCM 40, Uniform LBP 59, Geometry 3, Contour 64,\nFourier 16, HSV 192, Color Moments 9, Spatial HSV 288 (total = 671).\n"""\nfrom __future__ import annotations\nfrom dataclasses import dataclass, asdict\nimport cv2\nimport numpy as np\nfrom skimage.feature import local_binary_pattern\nfrom .preprocessing import PreparedImage\n\n@dataclass(frozen=True)\nclass FeatureParameters:\n    gray_levels: int = 32\n    glcm_distances: tuple[int, ...] = (1, 2)\n    glcm_offsets_unit: tuple[tuple[int,int], ...] = ((0,1),(-1,1),(-1,0),(-1,-1))\n    lbp_points: int = 8\n    lbp_radius: int = 1\n    contour_points: int = 64\n    fourier_resample: int = 256\n    fourier_coeffs: int = 16\n    hsv_bins: tuple[int,int,int] = (12,4,4)\n    spatial_bins: tuple[int,int,int] = (8,3,2)\n    spatial_grid: tuple[int,int] = (2,3)  # rows, columns\n\n    def dimensions(self):\n        p = self.lbp_points\n        # Number of non-rotation invariant uniform LBP labels: P(P-1)+3\n        return {\'glcm\':len(self.glcm_distances)*len(self.glcm_offsets_unit)*5,\n                \'lbp\':p*(p-1)+3,\n                \'geometry\':3,\n                \'contour\':self.contour_points,\n                \'fourier\':self.fourier_coeffs,\n                \'hsv\':int(np.prod(self.hsv_bins)),\n                \'moments\':9,\n                \'spatial\':int(np.prod(self.spatial_bins)*np.prod(self.spatial_grid))}\n\n    def config_for(self,name:str)->dict:\n        d=asdict(self)\n        fields={\'glcm\':(\'gray_levels\',\'glcm_distances\',\'glcm_offsets_unit\'),\n                \'lbp\':(\'lbp_points\',\'lbp_radius\'),\n                \'geometry\':(), \'contour\':(\'contour_points\',),\n                \'fourier\':(\'fourier_resample\',\'fourier_coeffs\'),\n                \'hsv\':(\'hsv_bins\',),\'moments\':(),\n                \'spatial\':(\'spatial_bins\',\'spatial_grid\')}\n        return {k:d[k] for k in fields[name]}\n\n\ndef masked_glcm(prep:PreparedImage, params:FeatureParameters)->np.ndarray:\n    q=(prep.gray.astype(np.int32)*params.gray_levels//256).astype(np.int16)\n    h,w=q.shape\n    mask=prep.mask\n    L=params.gray_levels\n    i_idx=np.arange(L,dtype=np.float64)[:,None]\n    j_idx=np.arange(L,dtype=np.float64)[None,:]\n    diff=i_idx-j_idx\n    diff2=diff*diff\n    inv_homo=1.0/(1.0+diff2)\n    vals=[]\n    for d in params.glcm_distances:\n        for dr0,dc0 in params.glcm_offsets_unit:\n            dr,dc=dr0*d,dc0*d\n            r0,r1=max(0,-dr),min(h,h-dr)\n            c0,c1=max(0,-dc),min(w,w-dc)\n            valid=mask[r0:r1,c0:c1] & mask[r0+dr:r1+dr,c0+dc:c1+dc]\n            if not np.any(valid): raise ValueError(\'No valid fish pixel pairs for GLCM\')\n            a=q[r0:r1,c0:c1][valid].astype(np.int32)\n            b=q[r0+dr:r1+dr,c0+dc:c1+dc][valid].astype(np.int32)\n            mat=np.bincount(a*L+b,minlength=L*L).reshape(L,L).astype(np.float64)\n            mat/=mat.sum()\n            asm=float(np.sum(mat**2))\n            contrast=float(np.sum(mat*diff2))\n            homogeneity=float(np.sum(mat*inv_homo))\n            p_i=mat.sum(axis=1); p_j=mat.sum(axis=0)\n            mi=float(np.dot(np.arange(L),p_i)); mj=float(np.dot(np.arange(L),p_j))\n            si=np.sqrt(float(np.dot((np.arange(L)-mi)**2,p_i)))\n            sj=np.sqrt(float(np.dot((np.arange(L)-mj)**2,p_j)))\n            corr=float(np.sum(mat*(i_idx-mi)*(j_idx-mj))/(si*sj)) if si>1e-12 and sj>1e-12 else 1.0\n            nonzero=mat[mat>0]\n            entropy=float(-np.sum(nonzero*np.log2(nonzero)))\n            vals.extend((asm,contrast,homogeneity,corr,entropy))\n    return np.array(vals,dtype=np.float32)\n\n\ndef uniform_lbp(prep:PreparedImage, params:FeatureParameters)->np.ndarray:\n    if params.lbp_points!=8:\n        # Function supports other P, but labels are P*(P-1)+3; code paths tested for P=8.\n        raise ValueError(\'Current validated implementation requires P=8\')\n    codes=local_binary_pattern(prep.gray,params.lbp_points,params.lbp_radius,method=\'nri_uniform\')\n    k=2*params.lbp_radius+1\n    valid=cv2.erode(prep.mask.astype(np.uint8),np.ones((k,k),np.uint8))>0\n    n=params.lbp_points*(params.lbp_points-1)+3\n    hist=np.bincount(codes[valid].astype(np.int32),minlength=n).astype(np.float64)[:n]\n    if hist.sum()==0: raise ValueError(\'No valid foreground positions for LBP\')\n    return (hist/hist.sum()).astype(np.float32)\n\n\ndef geometry(prep:PreparedImage)->np.ndarray:\n    x0,y0,x1,y1=prep.bbox\n    w,h=x1-x0,y1-y0\n    convex=cv2.convexHull(prep.contour)\n    hull_area=float(cv2.contourArea(convex))\n    solidity=prep.area/hull_area if hull_area>0 else 0.0\n    # Pixel mask area / geometric hull area can slightly exceed 1 because of discretization;\n    # use a hull mask for a consistent pixel-domain solidity estimate.\n    hull_mask=np.zeros(prep.mask.shape,np.uint8)\n    cv2.drawContours(hull_mask,[convex],-1,1,-1)\n    hull_px=int(hull_mask.sum())\n    solidity=prep.area/hull_px if hull_px else 0.0\n    return np.asarray([w/h,solidity,prep.area/(w*h)],dtype=np.float32)\n\n\ndef resample_contour(contour:np.ndarray,n:int)->np.ndarray:\n    pts=contour.reshape(-1,2).astype(np.float64)\n    if len(pts)<3: raise ValueError(\'Contour too short\')\n    # Use the upper-leftmost of all leftmost contour points as a stable origin.\n    start=int(np.lexsort((pts[:,1],pts[:,0]))[0])\n    pts=np.roll(pts,-start,axis=0)\n    # A consistent clockwise orientation is important when comparing profiles.\n    signed=np.sum(pts[:,0]*np.roll(pts[:,1],-1)-np.roll(pts[:,0],-1)*pts[:,1])\n    if signed<0: pts=np.concatenate((pts[:1],pts[:0:-1]),axis=0)\n    edges=np.linalg.norm(np.roll(pts,-1,axis=0)-pts,axis=1)\n    cum=np.r_[0,np.cumsum(edges)]\n    perimeter=cum[-1]\n    if perimeter<=0: raise ValueError(\'Degenerate contour\')\n    at=np.linspace(0,perimeter,n,endpoint=False)\n    xy=np.vstack((np.interp(at,cum,np.r_[pts[:,0],pts[0,0]]),\n                  np.interp(at,cum,np.r_[pts[:,1],pts[0,1]]))).T\n    return xy\n\n\ndef contour_profile(prep:PreparedImage,params:FeatureParameters)->np.ndarray:\n    xy=resample_contour(prep.contour,params.contour_points)\n    centroid=np.mean(xy,axis=0)\n    x0,y0,x1,y1=prep.bbox\n    scale=max(x1-x0,y1-y0)\n    return (np.linalg.norm(xy-centroid,axis=1)/scale).astype(np.float32)\n\n\ndef fourier_descriptor(prep:PreparedImage,params:FeatureParameters)->np.ndarray:\n    xy=resample_contour(prep.contour,params.fourier_resample)\n    z=(xy[:,0]-xy[:,0].mean())+1j*(xy[:,1]-xy[:,1].mean())\n    ff=np.fft.fft(z)\n    # Absolute coefficients remove rotation and starting-point phase;\n    # use F1 as in report when well-conditioned; else use dominant first harmonic.\n    scale=max(abs(ff[1]),abs(ff[-1]),1e-12)\n    if params.fourier_coeffs+1>=len(ff): raise ValueError(\'Too many Fourier coefficients\')\n    return (np.abs(ff[2:2+params.fourier_coeffs])/scale).astype(np.float32)\n\n\ndef color_bin_ids(hsv:np.ndarray,bins:tuple[int,int,int])->np.ndarray:\n    h,s,v=(hsv[:,i].astype(np.int32) for i in range(3))\n    nh,ns,nv=bins\n    hb=np.minimum(h*nh//180,nh-1)\n    sb=np.minimum(s*ns//256,ns-1)\n    vb=np.minimum(v*nv//256,nv-1)\n    return (hb*ns+sb)*nv+vb\n\n\ndef hsv_histogram(prep:PreparedImage,bins:tuple[int,int,int])->np.ndarray:\n    n=int(np.prod(bins))\n    idx=color_bin_ids(prep.hsv[prep.mask],bins)\n    hist=np.bincount(idx,minlength=n).astype(np.float64)\n    return (hist/hist.sum()).astype(np.float32)\n\n\ndef color_moments(prep:PreparedImage)->np.ndarray:\n    rgb=prep.rgb[prep.mask].astype(np.float64)/255.\n    mean=rgb.mean(axis=0)\n    z=rgb-mean\n    sigma=np.sqrt(np.mean(z*z,axis=0))\n    # Signed cube-root of the third central moment (NOT standardized skewness).\n    tau=np.cbrt(np.mean(z*z*z,axis=0))\n    return np.r_[mean,sigma,tau].astype(np.float32)\n\n\ndef spatial_histogram(prep:PreparedImage,params:FeatureParameters)->np.ndarray:\n    rows,cols=params.spatial_grid\n    bins=params.spatial_bins\n    nb=int(np.prod(bins))\n    x0,y0,x1,y1=prep.bbox\n    yy,xx=np.nonzero(prep.mask)\n    region_r=np.minimum((yy-y0)*rows//(y1-y0),rows-1)\n    region_c=np.minimum((xx-x0)*cols//(x1-x0),cols-1)\n    regions=region_r*cols+region_c\n    colors=color_bin_ids(prep.hsv[prep.mask],bins)\n    counts=np.bincount(regions*nb+colors,minlength=rows*cols*nb).reshape(rows*cols,nb).astype(np.float64)\n    totals=counts.sum(axis=1,keepdims=True)\n    np.divide(counts,totals,out=counts,where=totals>0)\n    return counts.reshape(-1).astype(np.float32)\n\n\ndef extract(prep:PreparedImage,params:FeatureParameters|None=None)->dict[str,np.ndarray]:\n    p=params or FeatureParameters()\n    feats={\'glcm\':masked_glcm(prep,p),\n           \'lbp\':uniform_lbp(prep,p),\n           \'geometry\':geometry(prep),\n           \'contour\':contour_profile(prep,p),\n           \'fourier\':fourier_descriptor(prep,p),\n           \'hsv\':hsv_histogram(prep,p.hsv_bins),\n           \'moments\':color_moments(prep),\n           \'spatial\':spatial_histogram(prep,p)}\n    for name,vector in feats.items():\n        if vector.shape!=(p.dimensions()[name],):\n            raise AssertionError(f\'{name}: invalid length {len(vector)}\')\n        if not np.isfinite(vector).all():\n            raise ValueError(f\'{name}: NaN or infinity in descriptor\')\n    if not np.isclose(feats[\'lbp\'].sum(),1,atol=1e-5): raise ValueError(\'LBP histogram sum invalid\')\n    if not np.isclose(feats[\'hsv\'].sum(),1,atol=1e-5): raise ValueError(\'HSV histogram sum invalid\')\n    return feats\n',encoding='utf-8')
print('Created',p)

In [ ]:
p=WORK/'fishcbir/database.py'
p.parent.mkdir(parents=True,exist_ok=True)
p.write_text('"""SQLite storage: one feature vector per image and immutable configuration key."""\nimport hashlib,json,sqlite3\nimport numpy as np\nfrom .features import FeatureParameters\n\nVERSION=\'step3-v1\'\n\ndef config_key(name:str,params:dict)->str:\n    blob=json.dumps({\'version\':VERSION,\'name\':name,\'params\':params},sort_keys=True,separators=(\',\',\':\'))\n    return hashlib.sha256(blob.encode(\'utf-8\')).hexdigest()[:20]\n\ndef open_database(path:str):\n    con=sqlite3.connect(str(path))\n    con.execute(\'PRAGMA foreign_keys=ON\')\n    con.execute(\'PRAGMA journal_mode=WAL\')\n    con.executescript(\'\'\'\n    CREATE TABLE IF NOT EXISTS images (\n      image_id TEXT PRIMARY KEY,\n      zip_member TEXT NOT NULL,\n      original_filename TEXT,\n      sha256 TEXT NOT NULL UNIQUE,\n      width INTEGER NOT NULL,\n      height INTEGER NOT NULL,\n      bbox_x0 INTEGER,\n      bbox_y0 INTEGER,\n      bbox_x1 INTEGER,\n      bbox_y1 INTEGER,\n      mask_area INTEGER,\n      mask_ratio REAL,\n      mask_status TEXT,\n      created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP\n    );\n    CREATE TABLE IF NOT EXISTS feature_configs (\n      config_key TEXT PRIMARY KEY,\n      family TEXT NOT NULL,\n      dimensions INTEGER NOT NULL,\n      parameters_json TEXT NOT NULL,\n      version TEXT NOT NULL\n    );\n    CREATE TABLE IF NOT EXISTS image_features (\n      image_id TEXT NOT NULL,\n      config_key TEXT NOT NULL,\n      dtype TEXT NOT NULL DEFAULT \'float32\',\n      dimensions INTEGER NOT NULL,\n      data BLOB NOT NULL,\n      extracted_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,\n      PRIMARY KEY (image_id, config_key),\n      FOREIGN KEY(image_id) REFERENCES images(image_id),\n      FOREIGN KEY(config_key) REFERENCES feature_configs(config_key)\n    );\n    CREATE TABLE IF NOT EXISTS extraction_logs (\n      log_id INTEGER PRIMARY KEY AUTOINCREMENT,\n      image_id TEXT NOT NULL,\n      status TEXT NOT NULL,\n      elapsed_seconds REAL,\n      error_message TEXT,\n      created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP\n    );\n    CREATE INDEX IF NOT EXISTS idx_feature_config ON image_features(config_key);\n    CREATE INDEX IF NOT EXISTS idx_logs_image ON extraction_logs(image_id);\n    \'\'\')\n    return con\n\ndef register_configs(con,p:FeatureParameters):\n    configs={}\n    for name,dim in p.dimensions().items():\n        prm=p.config_for(name)\n        key=config_key(name,prm)\n        con.execute(\'\'\'INSERT OR IGNORE INTO feature_configs\n          (config_key,family,dimensions,parameters_json,version)\n          VALUES(?,?,?,?,?)\'\'\',(key,name,dim,json.dumps(prm,ensure_ascii=False,sort_keys=True),VERSION))\n        configs[name]=key\n    con.commit()\n    return configs\n\ndef put_image(con,id,filename,alias,sha,w,h,prep):\n    x0,y0,x1,y1=prep.bbox\n    con.execute(\'\'\'INSERT INTO images(image_id,zip_member,original_filename,sha256,width,height,\n      bbox_x0,bbox_y0,bbox_x1,bbox_y1,mask_area,mask_ratio,mask_status)\n      VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?)\n      ON CONFLICT(image_id) DO UPDATE SET\n      zip_member=excluded.zip_member,original_filename=excluded.original_filename,\n      sha256=excluded.sha256,width=excluded.width,height=excluded.height,\n      bbox_x0=excluded.bbox_x0,bbox_y0=excluded.bbox_y0,\n      bbox_x1=excluded.bbox_x1,bbox_y1=excluded.bbox_y1,\n      mask_area=excluded.mask_area,mask_ratio=excluded.mask_ratio,mask_status=excluded.mask_status\n    \'\'\',(id,filename,alias,sha,w,h,x0,y0,x1,y1,prep.area,prep.mask_ratio,prep.status))\n\ndef put_features(con,id,features,configs):\n    for name,v in features.items():\n        v=np.asarray(v,dtype=\'<f4\')\n        con.execute(\'\'\'INSERT OR REPLACE INTO image_features\n          (image_id,config_key,dtype,dimensions,data) VALUES(?,?,?,?,?)\'\'\',\n          (id,configs[name],\'float32\',len(v),sqlite3.Binary(v.tobytes())))\n\ndef load_feature_matrix(con,config_key):\n    rows=con.execute(\'\'\'SELECT image_id,dimensions,data FROM image_features\n      WHERE config_key=? ORDER BY image_id\'\'\',(config_key,)).fetchall()\n    if not rows: return [],np.zeros((0,0),dtype=np.float32)\n    return [r[0] for r in rows],np.stack([np.frombuffer(r[2],dtype=\'<f4\',count=r[1]).copy() for r in rows])\n',encoding='utf-8')
print('Created',p)

In [ ]:
p=WORK/'fishcbir/cli.py'
p.parent.mkdir(parents=True,exist_ok=True)
p.write_text('"""CLI: python -m fishcbir.cli extract --zip images.zip --db fish.db --out outputs"""\nimport argparse,csv,hashlib,io,json,re,time,zipfile,traceback\nfrom collections import Counter\nfrom pathlib import Path\nimport cv2\nimport numpy as np\nfrom .preprocessing import preprocess\nfrom .features import extract,FeatureParameters\nfrom .database import open_database,register_configs,put_image,put_features,load_feature_matrix\n\n\ndef entries(z:zipfile.ZipFile):\n    numeric={Path(n).stem:n for n in z.namelist() if re.fullmatch(r\'\\d{4}\',Path(n).stem) and n.lower().endswith(\'.png\')}\n    if not numeric: raise RuntimeError(\'No 4-digit PNG images found in the ZIP\')\n    return dict(sorted(numeric.items()))\n\ndef aliases(z:zipfile.ZipFile, numeric:dict):\n    # Pair the second copy (species_* names) by exact content hash; never use species as a relevance label.\n    named=[n for n in z.namelist() if n.endswith(\'.png\') and n not in numeric.values()]\n    byhash={hashlib.sha256(z.read(n)).digest():n for n in named}\n    return {id:byhash.get(hashlib.sha256(z.read(n)).digest()) for id,n in numeric.items()}\n\n\ndef draw_check(rgb,prep,feature_vectors,id,outdir):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    outdir=Path(outdir); outdir.mkdir(parents=True,exist_ok=True)\n    fig,ax=plt.subplots(2,3,figsize=(12,6))\n    ax[0,0].imshow(rgb);ax[0,0].set_title(f\'Original {id}\')\n    ax[0,1].imshow(prep.mask,cmap=\'gray\');ax[0,1].set_title(\'Foreground binary mask\')\n    overlay=rgb.copy();cv2.drawContours(overlay,[prep.contour],-1,(245,10,10),2)\n    x0,y0,x1,y1=prep.bbox;cv2.rectangle(overlay,(x0,y0),(x1-1,y1-1),(20,25,220),2)\n    ax[0,2].imshow(overlay);ax[0,2].set_title(\'Contour and bounding box\')\n    ax[1,0].plot(feature_vectors[\'lbp\']);ax[1,0].set_title(\'Uniform LBP 59-bin histogram\')\n    ax[1,1].plot(feature_vectors[\'hsv\']);ax[1,1].set_title(\'Global HSV color histogram\')\n    ax[1,2].plot(feature_vectors[\'contour\']);ax[1,2].set_title(\'Normalized contour profile\')\n    for a in ax[0]: a.axis(\'off\')\n    fig.tight_layout();fig.savefig(outdir/f\'features_{id}.png\',dpi=130);plt.close(fig)\n\n\ndef audit(zippath):\n    with zipfile.ZipFile(zippath) as z:\n        fs=[n for n in z.namelist() if n.lower().endswith(\'.png\')]\n        num=entries(z)\n        names=aliases(z,num)\n        resolutions=Counter(); modes=Counter(); channels=Counter()\n        for id,n in num.items():\n            buf=np.frombuffer(z.read(n),np.uint8);img=cv2.imdecode(buf,cv2.IMREAD_UNCHANGED)\n            if img is None: raise ValueError(f\'Unreadable {n}\')\n            resolutions[str((img.shape[1],img.shape[0]))]+=1\n            channels[img.shape[2] if img.ndim==3 else 1]+=1\n        summary={\'zip_entries\':len(z.namelist()),\'png_files\':len(fs),\n                 \'numbered_unique_images\':len(num),\n                 \'exact_duplicate_pairs_with_original_names\':sum(x is not None for x in names.values()),\n                 \'numeric_first\':next(iter(num)), \'numeric_last\':next(reversed(num)),\n                 \'resolutions\':dict(resolutions),\'channels\':dict(channels)}\n        return summary\n\n\ndef process(zippath,dbpath,outdir,limit=None,sample_ids=(\'0000\',\'0032\',\'0187\',\'0334\',\'0917\')):\n    """Resume-safe extraction. Export matrices from DB, not only current batch."""\n    outdir=Path(outdir);outdir.mkdir(parents=True,exist_ok=True)\n    p=FeatureParameters()\n    con=open_database(dbpath)\n    config_keys=register_configs(con,p)\n    failures=[]\n    with zipfile.ZipFile(zippath) as z:\n        num=entries(z)\n        aliases_=aliases(z,num)\n        items=list(num.items())[:limit] if limit else list(num.items())\n        placeholders=\',\'.join(\'?\'*len(config_keys))\n        sql=(\'SELECT i.image_id FROM images i JOIN image_features f ON f.image_id=i.image_id \'\n             \'WHERE f.config_key IN (\'+placeholders+\') GROUP BY i.image_id \'\n             \'HAVING COUNT(DISTINCT f.config_key)=?\')\n        done=set(r[0] for r in con.execute(sql,[*config_keys.values(),len(config_keys)]).fetchall())\n        print(f\'Resuming: {len(done)} images already extracted; {len(items)} in input\',flush=True)\n        for idx,(image_id,member) in enumerate(items,1):\n            if image_id in done: continue\n            start=time.perf_counter()\n            try:\n                data=z.read(member)\n                bgr=cv2.imdecode(np.frombuffer(data,np.uint8),cv2.IMREAD_COLOR)\n                if bgr is None: raise ValueError(f\'Failed to decode image {member}\')\n                rgb=cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB)\n                prep=preprocess(rgb)\n                feat=extract(prep,p)\n                put_image(con,image_id,member,aliases_.get(image_id),hashlib.sha256(data).hexdigest(),rgb.shape[1],rgb.shape[0],prep)\n                put_features(con,image_id,feat,config_keys)\n                if image_id in sample_ids:\n                    draw_check(rgb,prep,feat,image_id,outdir/\'figures\')\n                dur=time.perf_counter()-start\n                con.execute(\'INSERT INTO extraction_logs(image_id,status,elapsed_seconds) VALUES(?,?,?)\',(image_id,\'ok\',dur))\n            except Exception as e:\n                failures.append((image_id,str(e)))\n                con.execute(\'INSERT INTO extraction_logs(image_id,status,error_message) VALUES(?,?,?)\',\n                            (image_id,\'error\',str(e)))\n            if idx%25==0 or idx==len(items):\n                con.commit()\n                count=con.execute(\'SELECT COUNT(*) FROM images\').fetchone()[0]\n                print(f\'Input {idx}/{len(items)}: DB images={count}, new failures={len(failures)}\',flush=True)\n    con.commit()\n    rows=con.execute(\'SELECT image_id,original_filename,mask_area,mask_ratio,mask_status FROM images ORDER BY image_id\').fetchall()\n    all_ids=[r[0] for r in rows]\n    config_matrices={}\n    for name,key in config_keys.items():\n        group_ids,matrix=load_feature_matrix(con,key)\n        if group_ids!=all_ids:\n            raise RuntimeError(f\'Incomplete feature group {name}: {len(group_ids)} vectors for {len(all_ids)} images\')\n        config_matrices[name]=matrix\n    np.savez_compressed(outdir/\'feature_matrices.npz\',ids=np.array(all_ids,dtype=\'U4\'),**config_matrices)\n    g=config_matrices[\'geometry\']\n    latest_times={}\n    for iid,t in con.execute("SELECT image_id, elapsed_seconds FROM extraction_logs WHERE status=\'ok\' ORDER BY log_id"):\n        latest_times[iid]=t\n    qc=[]\n    for i,(image_id,alias,area,ratio,status) in enumerate(rows):\n        qc.append({\'image_id\':image_id,\'alias_filename\':alias,\'mask_area\':area,\n                   \'mask_ratio\':round(ratio,6),\'mask_status\':status,\n                   \'aspect_ratio\':round(float(g[i,0]),6),\n                   \'solidity\':round(float(g[i,1]),6),\n                   \'extent\':round(float(g[i,2]),6),\n                   \'elapsed_seconds\':round(latest_times[image_id],4) if image_id in latest_times else None})\n    if qc:\n        with (outdir/\'image_qc.csv\').open(\'w\',newline=\'\',encoding=\'utf-8-sig\') as f:\n            writer=csv.DictWriter(f,fieldnames=list(qc[0]));writer.writeheader();writer.writerows(qc)\n    with (outdir/\'feature_samples.csv\').open(\'w\',newline=\'\',encoding=\'utf-8-sig\') as f:\n        writer=csv.writer(f)\n        writer.writerow([\'image_id\']+[f\'{name}_{j:03d}\' for name,dim in p.dimensions().items() for j in range(dim)])\n        for sample in sample_ids:\n            if sample not in all_ids: continue\n            i=all_ids.index(sample)\n            writer.writerow([sample]+[round(float(x),7) for name in p.dimensions() for x in config_matrices[name][i]])\n    runtimes=list(latest_times.values())\n    summary={\'total_images_in_zip\':len(num),\n        \'success_count\':len(all_ids),\'failure_count_current_run\':len(failures),\n        \'failures_current_run\':failures,\'dimensions\':p.dimensions(),\n        \'total_dimensions\':sum(p.dimensions().values()),\n        \'mask_status_counts\':dict(Counter(a[\'mask_status\'] for a in qc)),\n        \'mask_area_fraction_min\':round(min((a[\'mask_ratio\'] for a in qc),default=0),6),\n        \'mask_area_fraction_max\':round(max((a[\'mask_ratio\'] for a in qc),default=0),6),\n        \'per_image_seconds_median\':round(float(np.median(runtimes)),4) if runtimes else None,\n        \'elapsed_seconds_sum\':round(float(np.sum(runtimes)),2),\n        \'configs\':config_keys,\'code_version\':\'step3-v1\',\n        \'note\':\'Descriptor extraction validation only; no human-perceptual retrieval evaluation.\'}\n    (outdir/\'run_summary.json\').write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding=\'utf-8\')\n    con.close()\n    print(json.dumps(summary,ensure_ascii=False,indent=2))\n    return summary\n\ndef main():\n    a=argparse.ArgumentParser(description=\'Fish CBIR step 3: descriptor extraction and SQLite DB\')\n    s=a.add_subparsers(dest=\'action\',required=True)\n    aud=s.add_parser(\'audit\');aud.add_argument(\'--zip\',required=True)\n    ex=s.add_parser(\'extract\');ex.add_argument(\'--zip\',required=True)\n    ex.add_argument(\'--db\',default=\'data/fish.db\');ex.add_argument(\'--out\',default=\'outputs\')\n    ex.add_argument(\'--limit\',type=int,default=None)\n    args=a.parse_args()\n    if args.action==\'audit\': print(json.dumps(audit(args.zip),indent=2,ensure_ascii=False))\n    elif args.action==\'extract\':\n        Path(args.db).parent.mkdir(parents=True,exist_ok=True)\n        process(args.zip,args.db,args.out,args.limit)\n\nif __name__==\'__main__\':main()\n',encoding='utf-8')
print('Created',p)

In [ ]:
# Find the attached input dataset. ZIP is read directly without extraction.
input_base=Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('/mnt/data')
candidates=sorted(input_base.rglob('images.zip'))
if candidates:
    dataset_zip=candidates[0]
else:
    possible=sorted(input_base.rglob('0000.png'))
    if not possible:
        raise FileNotFoundError('Upload/attach images.zip or a directory containing 0000.png through 0999.png')
    folder=possible[0].parent
    dataset_zip=WORK/'images.zip'
    with zipfile.ZipFile(dataset_zip,'w',zipfile.ZIP_STORED) as z:
        for f in folder.glob('*.png'):
            z.write(f,arcname='images/'+f.name)
print('Using:',dataset_zip)

In [ ]:
# STEP 3: extract all eight descriptor groups into SQLite and NPZ. Resume-safe.
env=os.environ.copy()
env['PYTHONPATH']=str(WORK)
command=[sys.executable,'-m','fishcbir.cli','extract','--zip',str(dataset_zip),
         '--db',str(WORK/'fish.db'),'--out',str(WORK/'outputs')]
subprocess.run(command,check=True,cwd=str(WORK),env=env)

In [ ]:
# Inspect the actual results (not illustrative numbers)
import sqlite3
con=sqlite3.connect(WORK/'fish.db')
print('Images:',con.execute('SELECT COUNT(*) FROM images').fetchone()[0])
print('Vectors:',con.execute('SELECT COUNT(*) FROM image_features').fetchone()[0])
print('SQLite integrity:',con.execute('PRAGMA integrity_check').fetchone()[0])
con.close()
print(json.dumps(json.loads((WORK/'outputs/run_summary.json').read_text(encoding='utf-8')),indent=2,ensure_ascii=False))
from IPython.display import display, Image
for p in sorted((WORK/'outputs/figures').glob('*.png'))[:3]:
    display(Image(filename=str(p),width=800))